# Clustering Lab

 
Based of the amazing work you did in the Movie Industry you've been recruited to the NBA! You are working as the VP of Analytics that helps support a head scout, Mr. Rooney, for the worst team in the NBA probably the Wizards. Mr. Rooney just heard about Data Science and thinks it can solve all the team's problems!!! He wants you to figure out a way to find players that are high performing but maybe not highly paid that you can steal to get the team to the playoffs! 

In this document you will work through a similar process that we did in class with the NBA data (NBA_Perf_22 and nba_salaries_22), merging them together. This is from 22-23 season, feel free to update to 2023-24 season if you want.
# Data Sources:

https://www.basketball-reference.com/leagues/NBA_2024_totals.html # reference for performance data
https://www.basketball-reference.com/contracts/players.html # reference for salary data


Details: 

- Determine a way to use clustering to estimate based on performance if 
players are under or over paid, generally. 

- Then select players you believe would be best for your team and explain why. Do so in three categories: 
    * Examples that are not good choices (3 or 4) 
    * Several options that are good choices (3 or 4)
    * Several options that could work, assuming you can't get the players in the good category (3 or 4)

- You will decide the cutoffs for each category, so you should be able to explain why you chose them.

- Provide a well commented and clean report of your findings in a separate notebook that can be presented to Mr. Rooney, keeping in mind he doesn't understand...anything. Include a rationale for variables you included in the model, details on your approach and a overview of the results with supporting visualizations. 


Hints:

- Salary is the variable you are trying to understand 
- When interpreting you might want to use graphs that include variables that are the most correlated with Salary
- You'll need to scale the variables before performing the clustering
- Be specific about why you selected the players that you did, more detail is better
- Use good coding practices, comment heavily, indent, don't use for loops unless totally necessary and create modular sections that align with some outcome. If necessary create more than one script,list/load libraries at the top and don't include libraries that aren't used. 
- Be careful for non-traditional characters in the players names, certain graphs won't work when these characters are included.


In [27]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
# Reading in the data
perf_data = pd.read_csv("../data/NBA_Perf_22.csv", encoding='latin1')
sal_data = pd.read_csv("../data/nba_salaries_22.csv")

In [3]:
perf_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 812 entries, 0 to 811
Data columns (total 29 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  812 non-null    object 
 1   Pos     812 non-null    object 
 2   Age     812 non-null    int64  
 3   Tm      812 non-null    object 
 4   G       812 non-null    int64  
 5   GS      812 non-null    int64  
 6   MP      812 non-null    float64
 7   FG      812 non-null    float64
 8   FGA     812 non-null    float64
 9   FG%     797 non-null    float64
 10  3P      812 non-null    float64
 11  3PA     812 non-null    float64
 12  3P%     740 non-null    float64
 13  2P      812 non-null    float64
 14  2PA     812 non-null    float64
 15  2P%     784 non-null    float64
 16  eFG%    797 non-null    float64
 17  FT      812 non-null    float64
 18  FTA     812 non-null    float64
 19  FT%     715 non-null    float64
 20  ORB     812 non-null    float64
 21  DRB     812 non-null    float64
 22  TR

In [4]:
merged_data = pd.merge(perf_data, sal_data, on='Player', how='inner')
merged_data.head()

,Player,Pos,Age,Tm,G,GS,MP,FG,FGA,FG%,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Salary
0,Precious Achiuwa,C,22,TOR,73,28,23.6,3.6,8.3,0.439,...,2.0,4.5,6.5,1.1,0.5,0.6,1.2,2.1,9.1,"$2,840,160"
1,Steven Adams,C,28,MEM,76,75,26.3,2.8,5.1,0.547,...,4.6,5.4,10.0,3.4,0.9,0.8,1.5,2.0,6.9,"$17,926,829"
2,Bam Adebayo,C,24,MIA,56,56,32.6,7.3,13.0,0.557,...,2.4,7.6,10.1,3.4,1.4,0.8,2.6,3.1,19.1,"$30,351,780"
3,Santi Aldama,PF,21,MEM,32,0,11.3,1.7,4.1,0.402,...,1.0,1.7,2.7,0.7,0.2,0.3,0.5,1.1,4.1,"$2,094,120"
4,Nickeil Alexander-Walker,SG,23,TOT,65,21,22.6,3.9,10.5,0.372,...,0.6,2.3,2.9,2.4,0.7,0.4,1.4,1.6,10.6,"$5,009,633"


In [5]:
# Dropping our duplicates
nba_data = merged_data.drop_duplicates()
nba_data = nba_data.drop('Tm', axis = 1)
nba_data = nba_data.dropna()

In [6]:
# Remove currency symbols and commas, then convert to float
nba_data['Salary'] = nba_data['Salary'].replace(r'[\$,]', '', regex=True)
nba_data['Salary'] = pd.to_numeric(nba_data['Salary'], errors='coerce')

In [7]:
# Scaling the data
numeric_vars = nba_data.select_dtypes(include=['float64', 'int64'])
scaler = MinMaxScaler()
scaled_numeric_vars = scaler.fit_transform(numeric_vars)
scaled_df = pd.DataFrame(scaled_numeric_vars, columns=numeric_vars.columns, index=numeric_vars.index)
nba_data[numeric_vars.columns] = scaled_df

In [8]:
#Run the clustering algo with your best guess for K
x = nba_data.iloc[:, 2:-1]
y = nba_data['Salary']
kmeans_nba = KMeans(n_clusters=5, random_state=67).fit(x)

In [9]:
#View the results
print(kmeans_nba.cluster_centers_) 
print(kmeans_nba.labels_)
print(kmeans_nba.inertia_)

[[0.25119617 0.44217024 0.04075738 0.27829574 0.10797448 0.14633741
  0.42056167 0.13508772 0.15459589 0.31933333 0.09350504 0.09867209
  0.33239766 0.56343284 0.05512465 0.0553539  0.63043478 0.11117468
  0.1127234  0.11280824 0.10201429 0.19298246 0.07988722 0.12116228
  0.20676692 0.11545756]
 [0.33495671 0.72398589 0.70949477 0.87853741 0.61612554 0.7147935
  0.47363628 0.48201058 0.51322751 0.35722619 0.52228977 0.52946593
  0.34726984 0.60361141 0.36691729 0.35991379 0.72727922 0.18995859
  0.3899644  0.34598735 0.47795414 0.50324675 0.17091837 0.51339286
  0.43974733 0.6455518 ]
 [0.3        0.73474427 0.69930314 0.73722449 0.4361039  0.40379147
  0.69445629 0.10984127 0.12942613 0.23865714 0.48784195 0.41515391
  0.48099048 0.73992537 0.21353383 0.24852217 0.53017777 0.5689441
  0.56395194 0.5982018  0.23201058 0.36883117 0.3744898  0.34464286
  0.52536443 0.40559846]
 [0.34500745 0.76108075 0.3797481  0.65398126 0.30119225 0.37957424
  0.42749878 0.37905282 0.39960768 0.361049

In [10]:
#Create a visualization of the results with 2 or 3 variables that you think will best
#differentiate the clusters
fig = px.scatter_3d(nba_data, x="PTS", y="eFG%", z="MP", color=kmeans_nba.labels_,
                    title="Aye vs. Nay vs. Other votes for Democrat-introduced bills")
fig.write_html("my_plot.html")

In [11]:
#Evaluate the quality of the clustering using total variance explained and silhouette scores

In [25]:
# Variance explained:

total_sum_squares = np.sum((x - np.mean(x))**2)
total = np.sum(total_sum_squares)

between_SSE = (total-kmeans_nba.inertia_)

Var_explained = between_SSE/total
print(Var_explained*100)

70.5158878348908


/home/vscode/.local/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:84: FutureWarning:

The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)



In [28]:
# Silhouette Method

silhouette_scores = []
for k in range(3, 11):
    kmeans_obj = KMeans(n_clusters=k, algorithm="lloyd", random_state=1).fit(x)
    silhouette_scores.append(silhouette_score(x, kmeans_obj.labels_))

best_nc = silhouette_scores.index(max(silhouette_scores))+2
silhouette_scores

[np.float64(0.18294607873583396),
 np.float64(0.18616308944964072),
 np.float64(0.16502126632302253),
 np.float64(0.160126184319006),
 np.float64(0.15099448005802024),
 np.float64(0.12798489993350362),
 np.float64(0.1288898718542961),
 np.float64(0.12900945811719017)]

In [30]:
#Determine the ideal number of clusters using the elbow method and the silhouette coefficient

wcss = []
for i in range(3, 11):
    kmeans_obj_nba = KMeans(n_clusters=i, random_state=1).fit(x)
    wcss.append(kmeans_obj_nba.inertia_)

elbow_data_nba = pd.DataFrame({"k": range(3, 11), "wcss": wcss})
elbow_data_nba

,k,wcss
0,3,221.467867
1,4,199.295186
2,5,182.815328
3,6,173.221696
4,7,161.749378
5,8,156.866179
6,9,152.278017
7,10,145.779995


In [31]:
#Visualize the results of the elbow method
fig = px.line(elbow_data_nba, x="k", y="wcss", title="Elbow Method for Optimal k")
fig.update_layout(xaxis_title="Number of Clusters (k)", yaxis_title="Within-Cluster Sum of Squares (WCSS)")
fig.write_html("my_plot_2.html")

In [16]:
#Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results
kmeans_nba_2 = KMeans(n_clusters=2, random_state=67).fit(x)
print(kmeans_nba_2.cluster_centers_) 
print(kmeans_nba_2.labels_)
print(kmeans_nba_2.inertia_)

[[0.30382775 0.5579922  0.14626123 0.43975564 0.20427632 0.24081754
  0.484375   0.18603801 0.20583108 0.31824671 0.18204087 0.17545071
  0.37380702 0.61486768 0.09397507 0.09706783 0.62744615 0.18921625
  0.20533079 0.20739787 0.1387366  0.26958732 0.1368656  0.17516447
  0.31162728 0.20691456]
 [0.33778966 0.72847575 0.66818109 0.82737628 0.52917409 0.6045287
  0.49481758 0.41931736 0.44578515 0.346      0.45000695 0.44433611
  0.36408715 0.62734124 0.28957688 0.29017354 0.69345719 0.24211424
  0.39142386 0.36395631 0.38259501 0.4560309  0.20074697 0.43096405
  0.44871282 0.54650238]]
[0 1 1 0 0 1 0 1 1 0 0 1 0 1 1 0 0 0 0 1 0 0 0 1 1 1 1 0 1 1 1 1 0 0 0 1 1
 0 1 1 0 0 0 0 0 1 0 0 0 1 0 1 1 0 0 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 1 0 0 0
 0 0 1 0 1 0 1 0 0 0 0 0 0 0 1 1 1 1 1 1 1 0 0 1 0 0 1 1 0 0 0 0 1 0 0 0 0
 1 1 1 0 1 0 0 1 0 0 0 0 1 0 0 0 0 1 1 0 0 0 0 0 0 1 0 0 1 0 1 1 0 1 1 1 1
 1 0 1 1 0 0 0 0 0 0 0 1 1 1 0 1 1 1 1 0 0 0 0 0 0 1 1 1 1 0 0 0 0 1 1 1 1
 1 0 0 0 0 0 1 1 0 0 0 1 0 0 0 

In [32]:
#Once again evaluate the quality of the clustering using total variance explained and silhouette scores
silhouette_scores_2 = []
kmeans_obj = KMeans(n_clusters=2, algorithm="lloyd", random_state=1).fit(x)
silhouette_scores.append(silhouette_score(x, kmeans_obj.labels_))

best_nc = silhouette_scores.index(max(silhouette_scores))+2
silhouette_scores_2

[]

In [18]:
wcss = []
for i in range(1, 11):
    kmeans_obj_nba = KMeans(n_clusters=i, random_state=1).fit(x)
    wcss.append(kmeans_obj_nba.inertia_)

elbow_data_nba = pd.DataFrame({"k": range(1, 11), "wcss": wcss})
elbow_data_nba

,k,wcss
0,1,398.749930
1,2,257.519782
2,3,221.467867
3,4,199.295186
4,5,182.815328
5,6,173.221696
6,7,161.749378
7,8,156.866179
8,9,152.278017
9,10,145.779995


In [19]:
nba_data['Cluster'] = kmeans_nba.labels_
nba_data['Performance_Score'] = (nba_data['PTS'] + nba_data['MP'] + nba_data['eFG%']) / 3
nba_data['Value_Index'] = nba_data['Performance_Score'] / nba_data['Salary']
print(nba_data.groupby('Cluster')[['Salary', 'Performance_Score']].mean())
nba_data['Value_Index'] = nba_data['Performance_Score'] / nba_data['Salary']

           Salary  Performance_Score
Cluster                             
0        0.051155           0.319062
1        0.481657           0.709234
2        0.311863           0.627583
3        0.176112           0.527623
4        0.127761           0.466804


In [20]:
# Calculate percentiles
lower_val_thresh = nba_data['Salary'].quantile(0.15)
upper_val_thresh = nba_data['Salary'].quantile(0.95)
mp_thresh = nba_data['MP'].quantile(0.10)

# Filter: Keep players BETWEEN 10th and 90th percentile in Value_Index
# AND above 10th percentile in MP
filtered_players = nba_data[
    (nba_data['Salary'] > lower_val_thresh) &
    (nba_data['Salary'] < upper_val_thresh) &
    (nba_data['MP'] > mp_thresh)].copy()


In [21]:
undervalued_ranked = filtered_players.sort_values(by='Value_Index', ascending=False)
undervalued_ranked.head()

,Player,Pos,Age,G,GS,MP,FG,FGA,FG%,3P,...,AST,STL,BLK,TOV,PF,PTS,Salary,Cluster,Performance_Score,Value_Index
28,Desmond Bane,SG,0.181818,0.925926,0.926829,0.768571,0.572727,0.654028,0.486940,0.666667,...,0.250000,0.545455,0.142857,0.312500,0.510204,0.581081,0.042827,1,0.676874,15.804813
305,Terance Mann,SF,0.272727,0.987654,0.402439,0.734286,0.336364,0.369668,0.529851,0.200000,...,0.240741,0.318182,0.107143,0.208333,0.428571,0.331081,0.038669,3,0.565943,14.635500
151,Daniel Gafford,C,0.181818,0.876543,0.646341,0.491429,0.327273,0.236967,0.919776,0.000000,...,0.083333,0.181818,0.500000,0.187500,0.469388,0.283784,0.038669,2,0.564996,14.611011
312,Garrison Mathews,SG,0.272727,0.790123,0.402439,0.668571,0.218182,0.303318,0.371269,0.466667,...,0.092593,0.409091,0.142857,0.125000,0.510204,0.304054,0.040113,3,0.541870,13.508427
412,Isaiah Roby,PF,0.181818,0.543210,0.341463,0.520000,0.300000,0.308057,0.585821,0.222222,...,0.148148,0.363636,0.285714,0.208333,0.469388,0.307432,0.038669,4,0.513373,13.276016


In [22]:
overvalued_ranked = filtered_players.sort_values(by='Value_Index', ascending=True)
overvalued_ranked.head()

,Player,Pos,Age,G,GS,MP,FG,FGA,FG%,3P,...,AST,STL,BLK,TOV,PF,PTS,Salary,Cluster,Performance_Score,Value_Index
384,Michael Porter Jr.,SF,0.181818,0.098765,0.109756,0.757143,0.336364,0.507109,0.296642,0.244444,...,0.175926,0.500000,0.071429,0.270833,0.346939,0.300676,0.642543,4,0.481959,0.750082
300,Kevin Love,C,0.636364,0.901235,0.048780,0.560000,0.363636,0.454976,0.429104,0.555556,...,0.203704,0.181818,0.071429,0.270833,0.265306,0.425676,0.601478,3,0.548086,0.911232
233,Jaren Jackson Jr.,PF,0.136364,0.950617,0.951220,0.697143,0.463636,0.597156,0.401119,0.355556,...,0.101852,0.409091,0.821429,0.354167,0.693878,0.516892,0.601556,1,0.576942,0.959082
417,D'Angelo Russell,PG,0.272727,0.790123,0.792683,0.831429,0.527273,0.677725,0.393657,0.600000,...,0.657407,0.454545,0.107143,0.520833,0.387755,0.577703,0.652210,1,0.657521,1.008143
168,Draymond Green,PF,0.545455,0.555556,0.536585,0.742857,0.227273,0.232227,0.606343,0.066667,...,0.648148,0.590909,0.392857,0.625000,0.591837,0.219595,0.536130,2,0.542210,1.011340


#Write up the results in a separate notebook with supporting visualizations and 
an overview of how and why you made the choices you did. This should be at least 
500 words and should be written for a non-technical audience.